
# 西北地区 1 月平均气温地图

本示例演示用 Cartopy 绘制中国西北地区气温空间分布填色图的标准流程：

1. 指定画布投影（GeoAxes）——决定“纸往哪张地图坐标架上铺”；
2. ``contourf`` 绘制气温填色场，**必须**\绑定 ``transform``；
3. 叠加地理要素（海岸线、国界、河流），图层顺序由底到顶；
4. ``set_extent`` 裁剪西北区域；
5. 添加 ``colorbar``、标题，并标注兰州观测站点。

数据来自项目配套的 ``./data/northwest_temp.nc``\（2024 年 1 月 30 天的
模拟格点场，经度 100°~110°E、纬度 30°~40°N）。文件不在时自动改用
结构一致的合成场，保证脚本在任何环境都能运行。

<div class="alert alert-info"><h4>Note</h4><p>本区域深处内陆，海岸线与国界大多落在图幅之外（河流可见：黄河上游
   “几”字弯正好穿过本区），但叠加地理要素是绘制任何地图的标准步骤。</p></div>


## 1. 导入库并配置中文字体
``ccrs`` 负责投影，``cfeature`` 负责地理矢量要素；再统一指定无衬线中文字体。



In [ ]:
import os

import matplotlib

matplotlib.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei", "WenQuanYi Zen Hei",
    "Noto Sans CJK SC", "Arial Unicode MS",
]
matplotlib.rcParams["axes.unicode_minus"] = False  # 正确显示负号

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## 2. 读取数据：优先项目配套 NetCDF，缺失时退回合成场
依次尝试三个候选路径：项目根目录运行、画廊构建目录运行、独立运行。



In [ ]:
NC_CANDIDATES = ["./data/northwest_temp.nc",
                 "../data/northwest_temp.nc",
                 "../../data/northwest_temp.nc"]

nc_path = next((p for p in NC_CANDIDATES if os.path.exists(p)), None)

if nc_path:
    ds = xr.open_dataset(nc_path)
    # 教学模拟数据：取值 6~18，按课程文档口径以 ℃ 使用
    temp = ds["temp"].mean(dim="time")        # 1 月 30 天平均 -> (lat, lon)
    lon, lat = ds["lon"].values, ds["lat"].values
    source = f"配套数据 {nc_path}"
else:
    lon = np.linspace(100, 110, 21)           # 与配套文件同结构
    lat = np.linspace(30, 40, 11)
    LON, LAT = np.meshgrid(lon, lat)
    temp = xr.DataArray(12 + 0.8 * (LAT - 35) - 0.15 * (LON - 105) ** 2,
                        dims=("lat", "lon"), coords={"lat": lat, "lon": lon})
    source = "合成场（未找到数据文件）"

LON, LAT = np.meshgrid(lon, lat)
print(f"数据来源：{source}")
print(f"气温范围：{float(temp.min()):.1f} ~ {float(temp.max()):.1f} ℃")

## 3. 绘制完整地图
投影、填色场、地理要素、裁剪、色标、标注在同一步完成——
对交互式绘图而言这些代码可以分散执行，但成图时必须全部就位。



In [ ]:
fig, ax = plt.subplots(figsize=(10, 6),
                       subplot_kw={"projection": ccrs.PlateCarree()})

# 气温填色场：transform 声明数据贴在等经纬度（PlateCarree）坐标架上
cf = ax.contourf(LON, LAT, temp, levels=16, cmap="RdYlBu_r",
                 transform=ccrs.PlateCarree())

# 地理要素：图层由底到顶——填色场 → 河流 → 国界 → 海岸线
ax.add_feature(cfeature.RIVERS, linewidth=0.5, color="#4488dd")
ax.add_feature(cfeature.BORDERS, linewidth=0.8, color="black")
ax.coastlines(linewidth=0.8, color="black")

# 裁剪到数据区域（[西经, 东经, 南纬, 北纬]，显式指定 crs）
ax.set_extent([99, 111, 29, 41], crs=ccrs.PlateCarree())

# 色标与标题
cbar = fig.colorbar(cf, shrink=0.85)
cbar.set_label("气温 (℃)", fontsize=11, rotation=0)
ax.set_title("西北地区 2024 年 1 月平均气温", fontsize=14)

# 兰州站标注：lon=103.83°E, lat=36.06°N；标点与文字都需携带 transform
ax.scatter(103.83, 36.06, c="black", marker="o", s=30,
           transform=ccrs.PlateCarree())
ax.text(104.2, 36.06, "兰州站", fontsize=9,
        transform=ccrs.PlateCarree())

# 经纬网：只保留左、下两侧标签，避免四周标签互相挤压
gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.6)
gl.top_labels = False
gl.right_labels = False

plt.show()